# 01 — Exploración del dataset (UCF-Crime, 3 clases)

Requiere haber corrido `vad pipeline` (ingesta + preprocesamiento). Solo lee salidas; no modifica datos.

In [ ]:
import json, os
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
RAW = Path("data/raw/ucf_crime")
PROC = Path("data/processed/ucf_crime_3class")
MAN = Path("data/manifests")

## 1. Versión del dataset crudo

In [ ]:
raw_v = json.loads((MAN / "raw_dataset_version.json").read_text(encoding="utf-8"))
print("SHA-256      :", raw_v["dataset_sha256"])
print("Descarga     :", raw_v["download_date"], "| Ingesta:", raw_v["ingested_at"])
print("Totales      :", raw_v["totals"])
print("Resolución   :", raw_v["resolution"]["resolutions_WxHxC"])
pd.DataFrame(raw_v["per_category"]).T.drop(columns="sha256")

## 2. Splits procesados

In [ ]:
dfs = [pd.read_json(PROC / "splits" / f"{s}.jsonl", lines=True) for s in ("train", "val", "test")]
df = pd.concat(dfs, ignore_index=True)
pd.crosstab(df["class_name"], df["split"], margins=True)

In [ ]:
ax = pd.crosstab(df["class_name"], df["split"])[["train", "val", "test"]].plot.bar(figsize=(7, 4), rot=0)
ax.set_title("Videos por clase y split"); ax.set_ylabel("videos"); plt.tight_layout()

In [ ]:
pd.crosstab(df["ucf_category"], df["class_name"])

## 3. Longitud y brillo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for cls, g in df.groupby("class_name"):
    axes[0].hist(g["n_frames"], bins=40, alpha=0.5, label=cls)
    axes[1].hist(g["brightness"], bins=40, alpha=0.5, label=cls)
axes[0].set_title("Frames por video"); axes[1].set_title("Brillo medio")
axes[0].legend(); plt.tight_layout()
df.groupby("class_name")[["n_frames", "brightness"]].describe().round(1)

## 4. Videos excluidos

In [ ]:
exc = pd.read_csv(PROC / "excluded_videos.csv")
exc.groupby(["source_split", "class_name", "reason"]).size().unstack(fill_value=0)

## 5. Muestra visual: 16 segmentos de un video

In [ ]:
row = df[df["split"] == "train"].sample(1, random_state=0).iloc[0]
idx = np.linspace(0, len(row["frames"]) - 1, 16, dtype=int)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, i in zip(axes.ravel(), idx):
    img = cv2.cvtColor(cv2.imread(str(RAW / row["frames"][i])), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(f"frame {row['frame_indices'][i]}", fontsize=8); ax.axis("off")
fig.suptitle(f"{row['video_id']} — {row['class_name']}"); plt.tight_layout()